# VinAI Translate baseline trên Kaggle

Notebook dùng cùng code, config, dataset và seed 67 với Colab. Trước khi chạy: bật **GPU** và **Internet** trong Notebook settings, sau đó upload `NLP_colab.zip` thành Kaggle Dataset/Input. Notebook cố định một GPU để effective batch không đổi nếu runtime có nhiều GPU.

## 1. Kiểm tra GPU và môi trường

Thành công khi CUDA khả dụng và cell in đúng tên GPU.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # giữ effective batch giống Colab/nhóm
import subprocess, sys
subprocess.run(["nvidia-smi"], check=False)
import torch
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Hãy bật GPU trong Kaggle Notebook settings.")
print("GPU dùng cho baseline:", torch.cuda.get_device_name(0))
print("Số GPU nhìn thấy bởi tiến trình:", torch.cuda.device_count())

## 2. Tìm và giải nén ZIP từ Kaggle Input

Notebook tự tìm file `.zip` dưới `/kaggle/input`. Nếu có nhiều ZIP, điền một phần tên file vào `ZIP_NAME_HINT`. ZIP được đọc từ Input chỉ-đọc và giải nén sang `/kaggle/working`.

In [ ]:
from pathlib import Path, PurePosixPath
import shutil, tempfile, zipfile

ZIP_NAME_HINT = "NLP_colab"  # đổi nếu tên ZIP trên Kaggle khác
input_root = Path("/kaggle/input")
zip_candidates = sorted(input_root.rglob("*.zip"))
if ZIP_NAME_HINT:
    zip_candidates = [p for p in zip_candidates if ZIP_NAME_HINT.casefold() in p.name.casefold()]
if len(zip_candidates) != 1:
    found = [str(p) for p in sorted(input_root.rglob("*.zip"))]
    raise ValueError(f"Cần chọn đúng một ZIP; hiện tìm thấy: {found}. Hãy sửa ZIP_NAME_HINT.")
zip_path = zip_candidates[0]
extract_dir = Path(tempfile.mkdtemp(prefix="nlp_kaggle_", dir="/kaggle/working"))
extract_root = extract_dir.resolve()
with zipfile.ZipFile(zip_path) as archive:
    for member in archive.infolist():
        normalized = member.filename.replace("\\", "/")
        relative = PurePosixPath(normalized)
        if relative.is_absolute() or ".." in relative.parts or (relative.parts and ":" in relative.parts[0]):
            raise ValueError(f"ZIP chứa đường dẫn không an toàn: {member.filename}")
        if not relative.parts:
            continue
        destination = extract_dir.joinpath(*relative.parts)
        if not destination.resolve().is_relative_to(extract_root):
            raise ValueError(f"ZIP chứa đường dẫn không an toàn: {member.filename}")
        if member.is_dir() or normalized.endswith("/"):
            destination.mkdir(parents=True, exist_ok=True)
        else:
            destination.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as source, destination.open("wb") as target:
                shutil.copyfileobj(source, target)

def is_project_root(path):
    return ((path / "requirements-training.txt").is_file()
            and (path / "src/vinai_baseline/cli.py").is_file()
            and (path / "configs/vinai").is_dir())

candidates = [extract_dir] if is_project_root(extract_dir) else []
candidates += [p.parent for p in extract_dir.rglob("requirements-training.txt")
               if p.parent != extract_dir and is_project_root(p.parent)]
candidates = sorted(set(candidates), key=lambda p: (len(p.parts), str(p)))
if not candidates:
    raise FileNotFoundError(f"Không tìm thấy project root trong {zip_path}")
PROJECT_DIR = candidates[0]
os.chdir(PROJECT_DIR)
print("ZIP:", zip_path)
print("Project:", PROJECT_DIR)

## 3. Cài dependencies

Kaggle cần bật Internet để cài đúng stack và tải model public từ Hugging Face. Baseline không dùng TorchAO nên notebook gỡ phiên bản cũ nếu xung đột với PEFT.

In [ ]:
requirement_file = (PROJECT_DIR / "requirements-training-colab.txt"
                    if (PROJECT_DIR / "requirements-training-colab.txt").exists()
                    else PROJECT_DIR / "requirements-training.txt")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirement_file)], check=True)
from importlib.metadata import PackageNotFoundError, version
from packaging.version import Version
try:
    torchao_version = version("torchao")
except PackageNotFoundError:
    torchao_version = None
if torchao_version and Version(torchao_version) <= Version("0.16.0"):
    print(f"Gỡ torchao {torchao_version} vì xung đột với PEFT...")
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
print("Dependencies:", requirement_file.name)

## 4. Hugging Face token (không bắt buộc)

Model public vẫn tải được không cần token. Nếu muốn hạn mức ổn định hơn, tạo Kaggle Secret tên `HF_TOKEN`.

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
        print("Đã nạp HF_TOKEN từ Kaggle Secrets.")
except Exception:
    print("Không có HF_TOKEN; tiếp tục với model public.")

## 5. Chọn thí nghiệm và resume

Giữ `RUN_FULL=False` để smoke trước. Khi resume từ output của phiên Kaggle cũ, attach output đó làm Input, điền `RUN_ID_OVERRIDE`, đặt `RESUME_FROM_CHECKPOINT='last'`, `RUN_FULL=True` và trỏ `RESUME_RUN_SOURCE` tới đúng thư mục run trong `/kaggle/input`.

In [ ]:
import json, re
from datetime import datetime, timezone
DIRECTION = "en2vi"
MODE = "lora"  # zero_shot | lora | full
RUN_FULL = False
MEMBER_TAG = "thuan"
EXPERIMENT_TAG = "baseline_v2_t4safe16_kaggle"
RUN_ID_OVERRIDE = ""
RESUME_FROM_CHECKPOINT = ""
RESUME_RUN_SOURCE = ""  # ví dụ /kaggle/input/previous-run/.../<RUN_ID>

if DIRECTION not in {"en2vi", "vi2en"}: raise ValueError("DIRECTION không hợp lệ")
if MODE not in {"zero_shot", "lora", "full"}: raise ValueError("MODE không hợp lệ")
for label, value in {"MEMBER_TAG": MEMBER_TAG, "EXPERIMENT_TAG": EXPERIMENT_TAG}.items():
    if not re.fullmatch(r"[a-z0-9_-]+", value): raise ValueError(f"{label} không hợp lệ")
if RUN_ID_OVERRIDE and not re.fullmatch(r"[a-z0-9_-]+", RUN_ID_OVERRIDE):
    raise ValueError("RUN_ID_OVERRIDE không hợp lệ")
if RESUME_FROM_CHECKPOINT and (not RUN_ID_OVERRIDE or not RUN_FULL or MODE == "zero_shot"):
    raise ValueError("Resume cần RUN_ID_OVERRIDE, RUN_FULL=True và mode lora/full")
CONFIG = PROJECT_DIR / "configs/vinai" / f"{DIRECTION}_{MODE}.json"
config_preview = json.loads(CONFIG.read_text(encoding="utf-8"))
DATASET_ID = config_preview["data"]["dataset_id"]
TRAIN_SEED = config_preview["seed"]
DATA_DIR = PROJECT_DIR / Path(config_preview["data"]["manifest_file"]).parent
RUN_STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_ID = RUN_ID_OVERRIDE or f"{DATASET_ID}__{DIRECTION}__{MODE}__seed{TRAIN_SEED}__{MEMBER_TAG}__{EXPERIMENT_TAG}__{RUN_STAMP}"
OUTPUT_DIR = Path("/kaggle/working/NLP_runs") / DATASET_ID / DIRECTION / MODE / RUN_ID
if RESUME_FROM_CHECKPOINT:
    source = Path(RESUME_RUN_SOURCE)
    if not source.is_dir() or source.name != RUN_ID:
        raise FileNotFoundError("RESUME_RUN_SOURCE phải là đúng thư mục RUN_ID đã attach vào Kaggle Input")
    shutil.copytree(source, OUTPUT_DIR, dirs_exist_ok=True)
required = [DATA_DIR / f"segments_dev_{DIRECTION}.jsonl"]
if MODE != "zero_shot": required.append(DATA_DIR / f"segments_train_{DIRECTION}.jsonl")
missing = [str(p) for p in required if not p.exists()]
if missing: raise FileNotFoundError("Thiếu dữ liệu: " + ", ".join(missing))
training = config_preview["training"]
print("Config:", CONFIG)
print("Run ID:", RUN_ID)
print("Output:", OUTPUT_DIR)
print("Profile: train batch", training.get("per_device_train_batch_size"),
      "x accumulation", training.get("gradient_accumulation_steps"),
      "| eval batch", training.get("per_device_eval_batch_size"),
      "| checkpoint samples", training.get("checkpoint_eval_samples"))

## 6. Audit và hàm chạy có log

Log được stream trực tiếp và lưu tại `/kaggle/working/vinai_cli_logs`. Preflight chặn ZIP cũ chưa có bản sửa LoRA.

In [ ]:
from time import perf_counter
def format_duration(seconds):
    total = round(seconds); hours, rem = divmod(total, 3600); minutes, seconds = divmod(rem, 60)
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"

def run_cli(command, *extra):
    args = [sys.executable, "-u", "-m", "src.vinai_baseline.cli", command,
            "--config", str(CONFIG), "--output-dir", str(OUTPUT_DIR), *map(str, extra)]
    started = perf_counter()
    print(f"\n[{datetime.now().astimezone().isoformat(timespec='seconds')}] BẮT ĐẦU")
    print("RUN:", " ".join(args), flush=True)
    env = os.environ.copy(); env["PYTHONUNBUFFERED"] = "1"
    log_dir = Path("/kaggle/working/vinai_cli_logs"); log_dir.mkdir(parents=True, exist_ok=True)
    log_path = log_dir / f"{command}_{datetime.now().strftime('%Y%m%dT%H%M%S%f')}.log"
    lines = []
    with log_path.open("w", encoding="utf-8") as log_file:
        process = subprocess.Popen(args, cwd=PROJECT_DIR, env=env, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                                   errors="replace", bufsize=1)
        for line in process.stdout:
            print(line, end="", flush=True); log_file.write(line); log_file.flush(); lines.append(line.rstrip())
        return_code = process.wait()
    elapsed = perf_counter() - started
    if return_code:
        tail = "\n".join(lines[-80:]) or "(không có log)"
        raise RuntimeError(f"Lệnh {command} lỗi code {return_code} sau {format_duration(elapsed)}\n{tail}\nLog: {log_path}") from None
    print("HOÀN THÀNH sau", format_duration(elapsed), "| Log:", log_path)
    return elapsed

collator_source = (PROJECT_DIR / "src/vinai_baseline/data.py").read_text(encoding="utf-8")
if "prepare_decoder_input_ids_from_labels" not in collator_source:
    raise RuntimeError("ZIP cũ, chưa có bản sửa labels/logits LoRA")
print("Preflight LoRA collator: OK")
run_cli("audit")

## 7. Smoke test

LoRA/full dùng 32 mẫu train và 16 validation; zero-shot dùng 16 validation. Smoke chỉ xác minh pipeline, không kết luận chất lượng.

In [ ]:
COMMAND = "evaluate" if MODE == "zero_shot" else "train"
if RESUME_FROM_CHECKPOINT:
    RESULT_DIR = OUTPUT_DIR
    print("Bỏ qua smoke vì đang resume.")
else:
    SMOKE_OUTPUT = Path("/kaggle/working/vinai_smoke") / RUN_ID
    full_output = OUTPUT_DIR
    try:
        OUTPUT_DIR = SMOKE_OUTPUT
        if MODE == "zero_shot":
            run_cli(COMMAND, "--max-validation-samples", "16")
        else:
            run_cli(COMMAND, "--max-train-samples", "32", "--max-validation-samples", "16")
            run_cli("verify", "--max-validation-samples", "16")
        RESULT_DIR = SMOKE_OUTPUT
    finally:
        OUTPUT_DIR = full_output
    print("Smoke hoàn thành:", RESULT_DIR)

## 8. Full run

Sau khi smoke pass, đổi `START_FULL_NOW=True` ngay trong cell dưới rồi chạy cell này; không chạy lại cell cấu hình nên `RUN_ID` được giữ nguyên. Khi resume, `RUN_FULL=True` từ cell cấu hình cũng tự kích hoạt full run. Output nằm trong `/kaggle/working/NLP_runs`; hãy **Save Version** hoặc tải ZIP kết quả ở cell cuối trước khi phiên kết thúc.

In [ ]:
START_FULL_NOW = False  # đổi True sau khi smoke pass; chỉ chạy lại cell này
if START_FULL_NOW:
    RUN_FULL = True
if RUN_FULL:
    COMMAND = "evaluate" if MODE == "zero_shot" else "train"
    resume_args = (["--resume-from-checkpoint", RESUME_FROM_CHECKPOINT] if RESUME_FROM_CHECKPOINT else [])
    run_cli(COMMAND, *resume_args)
    if MODE != "zero_shot": run_cli("verify")
    RESULT_DIR = OUTPUT_DIR
else:
    print("Chưa chạy full; đang dùng kết quả smoke:", RESULT_DIR)

## 9. Xem metric, ví dụ dịch và đóng gói output

Cell kiểm tra đủ prediction, hiển thị metric/mẫu dịch và tạo một ZIP trong Kaggle Output để tải về hoặc dùng làm Input cho phiên resume.

In [ ]:
import random, pandas as pd
from IPython.display import display
RESULT_DIR = Path(RESULT_DIR)
metrics_path = RESULT_DIR / "reports/validation_metrics.json"
if not metrics_path.exists(): raise FileNotFoundError("Run chưa có validation_metrics.json")
metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
prediction_path = RESULT_DIR / "predictions/validation_predictions.jsonl"
predictions = [json.loads(line) for line in prediction_path.read_text(encoding="utf-8").splitlines() if line.strip()]
expected = int(metrics["evaluated_records"])
if len(predictions) != expected: raise RuntimeError(f"Prediction {len(predictions)}/{expected}")
def metric(name): return metrics.get(f"validation_{name}", metrics.get(name))
summary = [(name, metric(name)) for name in ["bleu", "chrf", "empty_output_rate",
           "length_ratio_mean", "number_preservation_f1", "reference_preservation_f1", "runtime"]]
summary.append(("peak_vram_mb", metrics.get("peak_vram_mb")))
display(pd.DataFrame(summary, columns=["metric", "value"]))
indices = random.Random(67).sample(range(len(predictions)), min(5, len(predictions)))
display(pd.DataFrame([{k: predictions[i][k] for k in ["segment_id", "source", "prediction", "target"]} for i in indices]))
archive_base = Path("/kaggle/working") / f"{RUN_ID}_result"
archive_path = shutil.make_archive(str(archive_base), "zip", root_dir=RESULT_DIR)
print("Artifact đầy đủ:", len(predictions), "prediction")
print("ZIP kết quả:", archive_path)